# Phase 2b : Comparaison Grid Search et fusion

In [1]:
import json
import torch
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from collections import defaultdict
import config

from utils_indexation import SearchIndex
from utils_evaluation import evaluate_retrieval_gpu

/home/marion/MIRAGE_TFE/env_marion/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuration et Chargement

In [2]:
model_names_t2i = ["siglip", "blip", "convnext_clip"]
model_names_i2t = ["siglip", "blip", "convnext_clip", "eva_clip"]
METRIC_CIBLE = 'R@1'


prefix_test = "test_"
all_model_names = list(set(model_names_t2i + model_names_i2t))

with open(config.BEST_WEIGHTS_FILE, 'r') as f:
    BEST_WEIGHTS = json.load(f)
print(f"Poids MIRAGE chargés depuis {config.BEST_WEIGHTS_FILE}")

Poids MIRAGE chargés depuis ./data/flickr30k/grid_search/best_weights.json


## Chargement des vecteurs de Test et cibles

In [3]:
print("\nChargement des vecteurs de Test...")
txt_vecs_test, img_vecs_test = {}, {}

for name in all_model_names:
    img_vecs_test[name] = np.load(f"{config.INDEX_DIR}/{prefix_test}{name}_img_vecs.npy")
    txt_vecs_test[name] = np.load(f"{config.INDEX_DIR}/{prefix_test}{name}_txt_vecs.npy")

modele_ref = all_model_names[0]
idx_img = SearchIndex(0); idx_img.load_from_disk(f"{config.INDEX_DIR}/{prefix_test}{modele_ref}_img")
test_img_ids = idx_img.image_ids

idx_txt = SearchIndex(0); idx_txt.load_from_disk(f"{config.INDEX_DIR}/{prefix_test}{modele_ref}_txt")
test_txt_to_img_id = idx_txt.image_ids

test_img_id_to_idx = {img_id: idx for idx, img_id in enumerate(test_img_ids)}
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Cibles T2I ---
targets_t2i_test = np.array([test_img_id_to_idx[iid] for iid in test_txt_to_img_id])
targets_t2i_gpu = torch.tensor(targets_t2i_test.reshape(-1, 1), device=device)

# --- Cibles I2T ---
test_img_to_txt_indices = defaultdict(list)
for txt_idx, iid in enumerate(test_txt_to_img_id):
    test_img_to_txt_indices[test_img_id_to_idx[iid]].append(txt_idx)
targets_i2t_test = [test_img_to_txt_indices[i] for i in range(len(test_img_id_to_idx))]

max_len_i2t = max(len(t) for t in targets_i2t_test)
targets_i2t_padded = [t + [-1] * (max_len_i2t - len(t)) for t in targets_i2t_test]
targets_i2t_gpu = torch.tensor(targets_i2t_padded, device=device)


Chargement des vecteurs de Test...
  Chargé : ./data/flickr30k/index_sauvegardes/test_eva_clip_img_index.bin (1000 vecteurs)
  Chargé : ./data/flickr30k/index_sauvegardes/test_eva_clip_txt_index.bin (5000 vecteurs)


## Définition des Fonctions Génériques

In [4]:
def run_ablation_study(query_vecs, gallery_vecs, model_names, targets_gpu, best_weights, device):
    """Calcule toutes les méthodes de fusion de manière agnostique (fonctionne pour T2I et I2T)."""
    results = {}
    
    # --- 1. Modèles individuels ---
    for name in model_names:
        S = torch.tensor(query_vecs[name], device=device) @ torch.tensor(gallery_vecs[name], device=device).T
        results[f"{name} seul"] = evaluate_retrieval_gpu(S, targets_gpu)

    # --- 2. Addition Simple ---
    S_add = sum((1.0/len(model_names)) * (torch.tensor(query_vecs[n], device=device) @ torch.tensor(gallery_vecs[n], device=device).T) for n in model_names)
    results["Fusion : Addition Simple"] = evaluate_retrieval_gpu(S_add, targets_gpu)

    # --- 3. Multiplication ---
    S_mult = None
    for name in model_names:
        S = torch.tensor(query_vecs[name], device=device) @ torch.tensor(gallery_vecs[name], device=device).T
        S_norm = (S - S.min()) / (S.max() - S.min() + 1e-8)
        S_mult = S_norm if S_mult is None else S_mult * S_norm
    results["Fusion : Multiplication"] = evaluate_retrieval_gpu(S_mult, targets_gpu)

    # --- 4. Concaténation ---
    q_concat = np.concatenate([query_vecs[n] for n in model_names], axis=1)
    g_concat = np.concatenate([gallery_vecs[n] for n in model_names], axis=1)
    q_concat = q_concat / np.linalg.norm(q_concat, axis=1, keepdims=True)
    g_concat = g_concat / np.linalg.norm(g_concat, axis=1, keepdims=True)
    
    S_concat = torch.tensor(q_concat, device=device) @ torch.tensor(g_concat, device=device).T
    results["Fusion : Concaténation"] = evaluate_retrieval_gpu(S_concat, targets_gpu)

    # --- 5. Maximum (Max Pooling) ---
    S_max = None
    for name in model_names:
        S = torch.tensor(query_vecs[name], device=device) @ torch.tensor(gallery_vecs[name], device=device).T
        S_norm = (S - S.min()) / (S.max() - S.min() + 1e-8)
        S_max = S_norm if S_max is None else torch.maximum(S_max, S_norm)
    results["Fusion : Maximum"] = evaluate_retrieval_gpu(S_max, targets_gpu)

    # --- 6. RRF (Reciprocal Rank Fusion) ---
    k_rrf = 60 
    N_queries, N_gallery = query_vecs[model_names[0]].shape[0], gallery_vecs[model_names[0]].shape[0]
    S_rrf = torch.zeros((N_queries, N_gallery), device=device)
    
    for name in model_names:
        S = torch.tensor(query_vecs[name], device=device) @ torch.tensor(gallery_vecs[name], device=device).T
        sorted_indices = torch.argsort(S, dim=1, descending=True)
        ranks = torch.zeros_like(S)
        rank_values = torch.arange(1, S.shape[1] + 1, device=device, dtype=torch.float).unsqueeze(0).expand_as(sorted_indices)
        ranks.scatter_(1, sorted_indices, rank_values)
        S_rrf += 1.0 / (k_rrf + ranks)
    results["Fusion : Reciprocal Rank (RRF)"] = evaluate_retrieval_gpu(S_rrf, targets_gpu)

    # --- 7. MIRAGE (Grid Search) ---
    S_mirage = sum(w * (torch.tensor(query_vecs[n], device=device) @ torch.tensor(gallery_vecs[n], device=device).T) for n, w in best_weights.items() if w > 0)
    results["MIRAGE (Grid Search)"] = evaluate_retrieval_gpu(S_mirage, targets_gpu)

    return results

def display_and_save_results(results_dict, task_name, save_dir):
    """Formate, affiche en gras, et sauvegarde les résultats en CSV et MD."""
    print("\n" + "="*60)
    print(f"RÉSULTATS TEST - ABLATION STUDY (Tâche {task_name})")
    print("="*60)

    df = pd.DataFrame.from_dict(results_dict, orient='index')
    df = df[['R@1', 'R@5', 'R@10', 'mAP', 'NDCG']]

    # Formatage en gras pour le Markdown
    df_formatted = df.copy()
    for col in df_formatted.columns:
        max_val = df[col].max() 
        df_formatted[col] = df[col].apply(lambda x: f"**{x:.4f}**" if x == max_val else f"{x:.4f}")
        
    display(Markdown(df_formatted.to_markdown()))

    # Sauvegarde
    chemin_csv = f"{save_dir}/resultats_{task_name.lower()}_ablation.csv"
    chemin_md = f"{save_dir}/resultats_{task_name.lower()}_ablation.md"
    
    df.to_csv(chemin_csv, float_format="%.4f")
    df_formatted.to_markdown(chemin_md)

    print(f"Résultats {task_name} sauvegardés avec succès dans :")
    print(f"   - {chemin_csv}")
    print(f"   - {chemin_md}")

## Exécution de l'Évaluation

In [5]:
results_t2i = run_ablation_study(
    query_vecs=txt_vecs_test, 
    gallery_vecs=img_vecs_test, 
    model_names=model_names_t2i, 
    targets_gpu=targets_t2i_gpu, 
    best_weights=BEST_WEIGHTS['t2i'][METRIC_CIBLE], 
    device=device
)
display_and_save_results(results_t2i, task_name="T2I", save_dir=config.GRID_SEARCH_DIR)

results_i2t = run_ablation_study(
    query_vecs=img_vecs_test, 
    gallery_vecs=txt_vecs_test, 
    model_names=model_names_i2t, 
    targets_gpu=targets_i2t_gpu, 
    best_weights=BEST_WEIGHTS['i2t'][METRIC_CIBLE], 
    device=device
)
display_and_save_results(results_i2t, task_name="I2T", save_dir=config.GRID_SEARCH_DIR)


RÉSULTATS TEST - ABLATION STUDY (Tâche T2I)


|                                | R@1        | R@5        | R@10       | mAP        | NDCG       |
|:-------------------------------|:-----------|:-----------|:-----------|:-----------|:-----------|
| siglip seul                    | 0.8304     | 0.9612     | 0.9796     | 0.8889     | 0.9151     |
| blip seul                      | 0.8212     | 0.9608     | 0.9776     | 0.8828     | 0.9105     |
| convnext_clip seul             | 0.7900     | 0.9414     | 0.9724     | 0.8574     | 0.8906     |
| Fusion : Addition Simple       | 0.8608     | 0.9734     | 0.9880     | 0.9108     | 0.9322     |
| Fusion : Multiplication        | 0.8632     | 0.9728     | 0.9876     | 0.9118     | 0.9329     |
| Fusion : Concaténation         | 0.8608     | 0.9732     | 0.9880     | 0.9108     | 0.9322     |
| Fusion : Maximum               | 0.8424     | 0.9708     | 0.9850     | 0.8987     | 0.9229     |
| Fusion : Reciprocal Rank (RRF) | 0.8520     | 0.9702     | 0.9856     | 0.9043     | 0.9272     |
| MIRAGE (Grid Search)           | **0.8684** | **0.9744** | **0.9882** | **0.9159** | **0.9361** |

Résultats T2I sauvegardés avec succès dans :
   - ./data/flickr30k/grid_search/resultats_t2i_ablation.csv
   - ./data/flickr30k/grid_search/resultats_t2i_ablation.md

RÉSULTATS TEST - ABLATION STUDY (Tâche I2T)


|                                | R@1        | R@5        | R@10       | mAP        | NDCG       |
|:-------------------------------|:-----------|:-----------|:-----------|:-----------|:-----------|
| siglip seul                    | 0.9420     | 0.9970     | 0.9980     | 0.8565     | 0.9337     |
| blip seul                      | 0.9360     | 0.9950     | 0.9990     | 0.8335     | 0.9216     |
| convnext_clip seul             | 0.9170     | 0.9920     | 0.9970     | 0.8146     | 0.9106     |
| eva_clip seul                  | 0.8970     | 0.9850     | 0.9920     | 0.7971     | 0.8996     |
| Fusion : Addition Simple       | **0.9600** | **1.0000** | **1.0000** | 0.8829     | **0.9477** |
| Fusion : Multiplication        | 0.9590     | 0.9990     | **1.0000** | 0.8824     | 0.9472     |
| Fusion : Concaténation         | **0.9600** | **1.0000** | **1.0000** | 0.8827     | 0.9476     |
| Fusion : Maximum               | 0.9470     | 0.9960     | 0.9990     | 0.8633     | 0.9370     |
| Fusion : Reciprocal Rank (RRF) | 0.9590     | 0.9980     | **1.0000** | 0.8738     | 0.9431     |
| MIRAGE (Grid Search)           | **0.9600** | 0.9990     | **1.0000** | **0.8829** | 0.9476     |

Résultats I2T sauvegardés avec succès dans :
   - ./data/flickr30k/grid_search/resultats_i2t_ablation.csv
   - ./data/flickr30k/grid_search/resultats_i2t_ablation.md
